# nnU-Net 2D Brain Tumor Segmentation (PKG-MU-Glioma, 4-Channel)

This notebook trains and evaluates a **lightweight nnU-Net 2D** model for binary tumor segmentation using the **PKG-MU-Glioma** NIfTI dataset.

## Pipeline Overview

### 1) Dataset and Inputs
- Source: PKG-MU-Glioma timepoint folders (`PatientID_*/Timepoint_*`)
- Input channels (4): `t1n`, `t1c`, `t2w`, `t2f`
- Target: binary tumor mask from NIfTI mask volume (`mask > 0`)
- Training sample granularity: 2D axial slices
- **Subsampling**: 1 timepoint per patient (most recent) to keep training fast
- Split strategy: **patient-level** train/validation/test split

### 2) Model and Optimization
- Architecture: **nnU-Net 2D** — plain U-Net with nnU-Net design choices
  - Instance Normalization + Leaky ReLU
  - Strided convolutions for downsampling
  - Transposed convolutions for upsampling
  - **Deep supervision** from multiple decoder stages
- Loss: hybrid DiceCE + Focal with deep supervision weighting
- Optimizer: AdamW with ReduceLROnPlateau scheduler
- ~3M parameters (vs ~21M for UNet++ EfficientNet-B4)

### 3) Evaluation and Analysis
- Metrics: Dice and IoU
- Qualitative outputs: prediction overlays and sanity checks
- Additional tools: modality normalization checks and Grad-CAM explainability

## 1. Import Libraries

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
from PIL import Image
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from monai.losses import DiceCELoss, FocalLoss

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

## 2. Data Exploration

In [ ]:
# Set data paths (same source used by 3d-recon notebook)
DATA_DIR = Path('/kaggle/input/datasets/prishapgpg/pkg-mu/PKG - MU-Glioma-Post/MU-Glioma-Post')
print(f"Dataset directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")


def minmax_norm_2d(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = x.astype(np.float32)
    x_min = float(np.nanmin(x))
    x_max = float(np.nanmax(x))
    if x_max - x_min < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - x_min) / (x_max - x_min + eps)


def discover_timepoint_dirs(data_root: Path) -> list:
    timepoints = []
    for p in sorted(data_root.glob('PatientID_*')):
        if p.is_dir():
            for tp in sorted(p.glob('Timepoint_*')):
                if tp.is_dir():
                    timepoints.append(tp)
    if not timepoints:
        timepoints = sorted([p for p in data_root.rglob('Timepoint_*') if p.is_dir()])
    return timepoints


def find_modality_file(timepoint_dir: Path, token: str) -> Path:
    files = sorted(timepoint_dir.glob('*.nii'))
    if token == 'mask':
        candidates = [f for f in files if 'mask' in f.name.lower()]
    else:
        candidates = [f for f in files if token in f.name.lower()]
    if not candidates:
        raise FileNotFoundError(f"Could not find '{token}' in {timepoint_dir}")
    return candidates[0]


def load_timepoint_volumes(timepoint_dir: Path) -> dict:
    paths = {
        't1n': find_modality_file(timepoint_dir, 't1n'),
        't1c': find_modality_file(timepoint_dir, 't1c'),
        't2w': find_modality_file(timepoint_dir, 't2w'),
        't2f': find_modality_file(timepoint_dir, 't2f'),
        'mask': find_modality_file(timepoint_dir, 'mask'),
    }
    vols = {k: nib.load(str(p)).get_fdata(dtype=np.float32) for k, p in paths.items()}
    base_shape = vols['t1n'].shape
    for k, v in vols.items():
        if v.shape != base_shape:
            raise ValueError(f"Shape mismatch in {timepoint_dir}: {k} has {v.shape}, expected {base_shape}")
    return vols


def extract_patient_id_from_timepoint(tp_path: Path) -> str:
    m = re.search(r'(PatientID_\d+)', str(tp_path))
    if m:
        return m.group(1)
    return tp_path.parent.name


# ── Discover all timepoints ──────────────────────────────────────────────────
timepoint_dirs = discover_timepoint_dirs(DATA_DIR)
print(f"\nTotal timepoints found: {len(timepoint_dirs)}")

# ── Use ALL timepoints per patient (was: only the most recent) ───────────────
# Patient-level train/val/test splits below prevent leakage, so multiple
# timepoints from one patient stay in the same split. This effectively
# 2–4×s the training data without changing the model or augmentation.
from collections import defaultdict

patient_timepoints = defaultdict(list)
for tp in timepoint_dirs:
    pid = extract_patient_id_from_timepoint(tp)
    patient_timepoints[pid].append(tp)

subsampled_timepoints = []
for pid in sorted(patient_timepoints.keys()):
    subsampled_timepoints.extend(sorted(patient_timepoints[pid]))

tp_counts = [len(patient_timepoints[pid]) for pid in patient_timepoints]
print(f"Patients: {len(patient_timepoints)}")
print(f"All timepoints across patients: {len(subsampled_timepoints)}")
print(f"Timepoints per patient — min: {min(tp_counts)}, max: {max(tp_counts)}, mean: {np.mean(tp_counts):.2f}")

# ── Build per-slice sample records ───────────────────────────────────────────
# Each record is tagged with `has_tumor` so the data-prep cell can filter
# the massive empty-mask majority from training without losing it from eval.
sample_records = []
skipped_empty = 0
n_tumor = 0
n_no_tumor = 0

for tp in subsampled_timepoints:
    try:
        vols = load_timepoint_volumes(tp)
    except Exception as ex:
        print(f"Skipping {tp} due to loading error: {ex}")
        continue

    z_dim = vols['t1n'].shape[2]
    patient_id = extract_patient_id_from_timepoint(tp)

    for z in range(z_dim):
        # Skip slices where all 4 modality channels are essentially empty
        slice_energy = sum(
            np.abs(vols[mod][:, :, z]).sum() for mod in ('t1n', 't1c', 't2w', 't2f')
        )
        if slice_energy < 1.0:
            skipped_empty += 1
            continue

        has_tumor = bool(vols['mask'][:, :, z].sum() > 0)
        if has_tumor:
            n_tumor += 1
        else:
            n_no_tumor += 1

        sample_records.append({
            'timepoint_dir': str(tp),
            'patient_id': patient_id,
            'z': int(z),
            'has_tumor': has_tumor,
        })

print(f"Total slice samples: {len(sample_records)} (filtered {skipped_empty} blank-image slices)")
print(f"  Tumor slices:    {n_tumor} ({100*n_tumor/len(sample_records):.1f}%)")
print(f"  No-tumor slices: {n_no_tumor} ({100*n_no_tumor/len(sample_records):.1f}%)")

In [ ]:
# Visualize sample multimodal slices from PKG-MU source
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    idx = np.random.randint(0, len(sample_records))
    rec = sample_records[idx]
    vols = load_timepoint_volumes(Path(rec['timepoint_dir']))
    z = rec['z']

    # Use FLAIR-equivalent channel (t2f) for top row quick preview
    img = minmax_norm_2d(vols['t2f'][:, :, z])
    mask = (vols['mask'][:, :, z] > 0).astype(np.float32)

    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f"Sample {i+1} - T2-FLAIR (z={z})")
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='gray')
    axes[1, i].set_title(f"Sample {i+1} - Mask")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

# Check data dimensions from one random record
rec0 = sample_records[0]
vols0 = load_timepoint_volumes(Path(rec0['timepoint_dir']))
print(f"\nVolume shape (H, W, Z): {vols0['t1n'].shape}")
print(f"Unique mask values (sample z): {np.unique((vols0['mask'][:, :, rec0['z']] > 0).astype(np.float32))}")

## 3. Custom Dataset Class

In [ ]:
class BrainTumorDataset(Dataset):
    """Custom Dataset for PKG-MU NIfTI slices with 4-channel multimodal input."""

    def __init__(self, sample_records, transform=None, img_size=256, patch_transform=None):
        self.sample_records = sample_records
        self.transform = transform
        self.img_size = img_size
        self.patch_transform = patch_transform

        # Eagerly pre-load all volumes before DataLoader workers are forked.
        # On Linux (fork start method), workers inherit this cache via copy-on-write —
        # no per-worker volume duplication. Prevents OOM kills from lazy loading.
        unique_dirs = sorted({r['timepoint_dir'] for r in sample_records})
        print(f"  Pre-loading {len(unique_dirs)} timepoint volume(s)...")
        self._cache = {d: load_timepoint_volumes(Path(d)) for d in unique_dirs}
        print(f"  Volume cache ready ({len(self._cache)} timepoints).")

    def __len__(self):
        return len(self.sample_records)

    def _get_vols(self, timepoint_dir: str) -> dict:
        return self._cache[timepoint_dir]

    def _build_image_mask(self, idx):
        rec = self.sample_records[idx]
        vols = self._get_vols(rec['timepoint_dir'])
        z = int(rec['z'])

        # 4-channel input: t1n, t1c, t2w, t2f
        ch_t1n = minmax_norm_2d(vols['t1n'][:, :, z])
        ch_t1c = minmax_norm_2d(vols['t1c'][:, :, z])
        ch_t2w = minmax_norm_2d(vols['t2w'][:, :, z])
        ch_t2f = minmax_norm_2d(vols['t2f'][:, :, z])
        image = np.stack([ch_t1n, ch_t1c, ch_t2w, ch_t2f], axis=-1).astype(np.float32)  # [H,W,4]

        # Binary tumor mask
        mask = (vols['mask'][:, :, z] > 0).astype(np.float32)  # [H,W]

        return image, mask

    def get_raw_item(self, idx):
        """Get raw sample without transforms (for sampler analysis)."""
        image, mask = self._build_image_mask(idx)
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image = torch.from_numpy(image).permute(2, 0, 1)  # [4,H,W]
        mask = torch.from_numpy(mask).unsqueeze(0)         # [1,H,W]
        return image, mask

    def __getitem__(self, idx):
        image, mask = self._build_image_mask(idx)

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask'].unsqueeze(0)
        else:
            image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
            image = torch.from_numpy(image).permute(2, 0, 1)
            mask = torch.from_numpy(mask).unsqueeze(0)

        # Patch-based training sampler (2D equivalent of volume patching)
        if self.patch_transform is not None:
            sample = {'image': image, 'label': mask}
            patch_samples = self.patch_transform(sample)
            if isinstance(patch_samples, list):
                patch = patch_samples[np.random.randint(0, len(patch_samples))]
            else:
                patch = patch_samples
            image = patch['image']
            mask = patch['label']

        return image, mask


## 4. nnU-Net 2D Model Definition

Lightweight U-Net following nnU-Net design principles:
- **InstanceNorm2d + LeakyReLU** (better for small batches than BatchNorm)
- **Strided convolutions** for downsampling (learns the downsampling)
- **Transposed convolutions** for upsampling
- **Deep supervision** from decoder stages 1–3 (multi-scale loss)
- **~3M parameters** — ~7× lighter than the previous UNet++ EfficientNet-B4

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  nnU-Net 2D — lightweight encoder-decoder with deep supervision
# ──────────────────────────────────────────────────────────────────────────────

def _num_groups(channels: int, target: int = 8) -> int:
    """Pick the largest divisor of `channels` that is <= `target`."""
    for g in range(min(target, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


class ConvBlock(nn.Module):
    """Two 3x3 convolutions with GroupNorm + LeakyReLU."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class nnUNet2D(nn.Module):
    """
    Compact U-Net with nnU-Net design choices for 2D segmentation.

    v3 changes (anti-overfitting):
    - Halved channel widths: [16, 32, 64, 128, 128] (~700K params vs 2.8M)
    - Dropout2d in BOTH encoder and decoder (full-path regularization)
    - Higher default drop_rate (0.3)
    """

    def __init__(self, in_channels=4, num_classes=1, deep_supervision=True,
                 drop_rate=0.3):
        super().__init__()
        self.deep_supervision = deep_supervision
        feats = [16, 32, 64, 128, 128]

        # ── Encoder (with dropout after downsampling) ────────────────────
        self.enc1 = ConvBlock(in_channels, feats[0])
        self.down1 = nn.Conv2d(feats[0], feats[0], 2, stride=2, bias=False)
        self.enc_drop1 = nn.Dropout2d(drop_rate * 0.5)  # lighter at shallow levels

        self.enc2 = ConvBlock(feats[0], feats[1])
        self.down2 = nn.Conv2d(feats[1], feats[1], 2, stride=2, bias=False)
        self.enc_drop2 = nn.Dropout2d(drop_rate * 0.5)

        self.enc3 = ConvBlock(feats[1], feats[2])
        self.down3 = nn.Conv2d(feats[2], feats[2], 2, stride=2, bias=False)
        self.enc_drop3 = nn.Dropout2d(drop_rate)

        self.enc4 = ConvBlock(feats[2], feats[3])
        self.down4 = nn.Conv2d(feats[3], feats[3], 2, stride=2, bias=False)
        self.enc_drop4 = nn.Dropout2d(drop_rate)

        # ── Bottleneck ───────────────────────────────────────────────────
        self.bottleneck = ConvBlock(feats[3], feats[4])
        self.bn_drop = nn.Dropout2d(drop_rate)

        # ── Decoder (with dropout) ───────────────────────────────────────
        self.up4 = nn.ConvTranspose2d(feats[4], feats[3], 2, stride=2)
        self.dec4 = ConvBlock(feats[3] * 2, feats[3])
        self.drop4 = nn.Dropout2d(drop_rate)

        self.up3 = nn.ConvTranspose2d(feats[3], feats[2], 2, stride=2)
        self.dec3 = ConvBlock(feats[2] * 2, feats[2])
        self.drop3 = nn.Dropout2d(drop_rate)

        self.up2 = nn.ConvTranspose2d(feats[2], feats[1], 2, stride=2)
        self.dec2 = ConvBlock(feats[1] * 2, feats[1])
        self.drop2 = nn.Dropout2d(drop_rate * 0.5)

        self.up1 = nn.ConvTranspose2d(feats[1], feats[0], 2, stride=2)
        self.dec1 = ConvBlock(feats[0] * 2, feats[0])

        # ── Output heads ─────────────────────────────────────────────────
        self.seg_out = nn.Conv2d(feats[0], num_classes, 1)

        if deep_supervision:
            self.ds3 = nn.Conv2d(feats[2], num_classes, 1)
            self.ds2 = nn.Conv2d(feats[1], num_classes, 1)

    def forward(self, x):
        # ── Encoder ──────────────────────────────────────────────────────
        e1 = self.enc1(x)
        e2 = self.enc2(self.enc_drop1(self.down1(e1)))
        e3 = self.enc3(self.enc_drop2(self.down2(e2)))
        e4 = self.enc4(self.enc_drop3(self.down3(e3)))

        # ── Bottleneck ───────────────────────────────────────────────────
        bn = self.bn_drop(self.bottleneck(self.enc_drop4(self.down4(e4))))

        # ── Decoder ──────────────────────────────────────────────────────
        d4 = self.drop4(self.dec4(torch.cat([self.up4(bn), e4], dim=1)))
        d3 = self.drop3(self.dec3(torch.cat([self.up3(d4), e3], dim=1)))
        d2 = self.drop2(self.dec2(torch.cat([self.up2(d3), e2], dim=1)))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        out = self.seg_out(d1)

        if self.training and self.deep_supervision:
            return out, self.ds2(d2), self.ds3(d3)

        return out


# Sanity check
_test_model = nnUNet2D(in_channels=4, num_classes=1, deep_supervision=True)
_test_input = torch.randn(1, 4, 192, 192)
_test_model.train()
_outs = _test_model(_test_input)
print(f"nnU-Net 2D v3 (compact, full-path dropout):")
print(f"  Input:  {_test_input.shape}")
for i, o in enumerate(_outs):
    print(f"  DS level {i}: {o.shape}")
_test_model.eval()
print(f"  Eval output: {_test_model(_test_input).shape}")
_n = sum(p.numel() for p in _test_model.parameters())
print(f"  Parameters: {_n:,}")
del _test_model, _test_input, _outs

In [ ]:
class OptimalBrainTumorLoss(nn.Module):
    """Hybrid loss for brain tumor segmentation using MONAI — 70% DiceCE + 30% Focal."""
    def __init__(self):
        super().__init__()
        self.dice_ce = DiceCELoss(
            sigmoid=True, squared_pred=True, lambda_dice=0.6, lambda_ce=0.4
        )
        self.focal = FocalLoss(gamma=2.0, alpha=0.75)

    def forward(self, pred, target):
        return 0.7 * self.dice_ce(pred, target) + 0.3 * self.focal(pred, target)


class DeepSupervisionLoss(nn.Module):
    """
    Wraps a base loss to support deep supervision outputs from nnU-Net.

    During training the model returns (full_res, half_res, quarter_res).
    Each auxiliary output is upsampled to full resolution and weighted:
        full_res: 1.0,  half_res: 0.5,  quarter_res: 0.25
    """
    def __init__(self, base_loss, weights=(1.0, 0.5, 0.25)):
        super().__init__()
        self.base_loss = base_loss
        self.weights = weights

    def forward(self, preds, target):
        if isinstance(preds, tuple):
            total = 0.0
            for w, p in zip(self.weights, preds):
                if p.shape[-2:] != target.shape[-2:]:
                    p = F.interpolate(p, size=target.shape[-2:], mode='bilinear', align_corners=False)
                total += w * self.base_loss(p, target)
            return total / sum(self.weights[:len(preds)])
        else:
            return self.base_loss(preds, target)


def dice_coefficient(pred, target, threshold=0.5):
    """Calculate Dice coefficient.

    Handles the empty-slice edge case correctly:
    - Both pred and GT empty → Dice = 1.0 (correct true negative)
    - Only one empty → Dice = 0.0 (incorrect prediction)
    """
    pred = (torch.sigmoid(pred) > threshold).float()

    pred_sum = pred.sum()
    target_sum = target.sum()

    # Both empty: model correctly predicted "no tumor" → perfect score
    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = (pred * target).sum()
    dice = (2.0 * intersection) / (pred_sum + target_sum + 1e-6)
    return dice.item()


def iou_score(pred, target, threshold=0.5):
    """Calculate Intersection over Union (IoU).

    Same empty-slice handling as dice_coefficient.
    """
    pred = (torch.sigmoid(pred) > threshold).float()

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = (pred * target).sum()
    union = pred_sum + target_sum - intersection
    iou = intersection / (union + 1e-6)
    return iou.item()


class EmptySlicePenalty(nn.Module):
    """Per-sample false-positive volume penalty for empty-GT slices.

    For each sample in the batch whose ground-truth mask is empty, adds a
    penalty proportional to the *mean predicted probability* across the slice.
    On slices that contain tumor, this term is zero — so it never fights with
    the main Dice/CE objective on positive examples.

    The penalty is applied only to the full-resolution prediction (not the
    deep-supervision auxiliary heads).
    """

    def __init__(self, weight: float = 0.1):
        super().__init__()
        self.weight = float(weight)

    def forward(self, full_res_logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # full_res_logits, target: (B, 1, H, W)
        prob = torch.sigmoid(full_res_logits)
        empty_mask = (target.sum(dim=(1, 2, 3)) == 0).float()           # (B,)
        if empty_mask.sum() == 0:
            return full_res_logits.new_zeros(())
        mean_fp = prob.mean(dim=(1, 2, 3))                              # (B,)
        penalty = (mean_fp * empty_mask).sum() / empty_mask.sum().clamp(min=1.0)
        return self.weight * penalty


class CriterionWithEmptyPenalty(nn.Module):
    """Wraps any criterion (including DeepSupervisionLoss) and adds the
    empty-slice penalty on the full-resolution output only.
    """

    def __init__(self, base_criterion: nn.Module, penalty_weight: float = 0.1):
        super().__init__()
        self.base = base_criterion
        self.empty_penalty = EmptySlicePenalty(weight=penalty_weight)

    def forward(self, preds, target):
        base_loss = self.base(preds, target)
        full_res = preds[0] if isinstance(preds, (tuple, list)) else preds
        return base_loss + self.empty_penalty(full_res, target)



In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  EMA + TTA-aware validation utilities
#
#  Two regularizers that work together:
#    1. ModelEMA — exponential moving average of weights. Smooths the noisy
#       SGD trajectory; the EMA model usually generalizes better than the
#       final SGD weights and shrinks the train/val gap.
#    2. validate_with_tta_postproc — runs validation with the same TTA +
#       morphological post-processing pipeline used at test time, so model
#       selection picks the checkpoint that's best at *deployment*, not the
#       one that's best on raw single-pass predictions.
# ──────────────────────────────────────────────────────────────────────────────
import copy
import ttach as tta_lib
from scipy import ndimage as _ndimage


class ModelEMA:
    """Exponential moving average of model weights.

    Maintains a shadow copy whose params evolve as:
        ema_param = decay * ema_param + (1 - decay) * model_param
    """

    def __init__(self, model, decay=0.999):
        unwrapped = model.module if isinstance(model, nn.DataParallel) else model
        self.module = copy.deepcopy(unwrapped).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        unwrapped = model.module if isinstance(model, nn.DataParallel) else model
        msd = unwrapped.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1.0 - self.decay)
            else:
                v.copy_(msd[k])


def post_process_mask(pred_prob, threshold=0.5):
    """Clean a probability map with morphology + largest-component selection."""
    pred_binary = (pred_prob > threshold).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    pred_binary = cv2.morphologyEx(pred_binary, cv2.MORPH_OPEN, kernel)
    pred_binary = cv2.morphologyEx(pred_binary, cv2.MORPH_CLOSE, kernel)
    pred_binary = _ndimage.binary_fill_holes(pred_binary).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_binary, connectivity=8)
    if num_labels > 1:
        largest = int(np.argmax(stats[1:, cv2.CC_STAT_AREA])) + 1
        pred_binary = (labels == largest).astype(np.uint8)
    return pred_binary


# Shared TTA configuration (used during training validation AND final test eval)
TTA_TRANSFORMS = tta_lib.Compose([
    tta_lib.HorizontalFlip(),
    tta_lib.VerticalFlip(),
    tta_lib.Rotate90(angles=[0, 90, 180, 270]),
])


class _DSStripWrapper(nn.Module):
    """Wraps a deep-supervision model so that TTA only sees the full-res tensor."""

    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x):
        out = self.base(x)
        return out[0] if isinstance(out, (tuple, list)) else out


def validate_with_tta_postproc(eval_module, dataloader, device, criterion=None):
    """Validation using TTA averaging + morphological post-processing.

    The metric matches deployment, so model selection picks the checkpoint
    that performs best AFTER the test-time pipeline — not just on raw logits.

    Returns: (mean_loss_or_nan, mean_dice, mean_iou)
    """
    base = eval_module.module if isinstance(eval_module, nn.DataParallel) else eval_module
    base.eval()
    wrapped = _DSStripWrapper(base).to(device)
    tta_model = tta_lib.SegmentationTTAWrapper(wrapped, TTA_TRANSFORMS, merge_mode='mean')

    dice_scores, iou_scores, losses = [], [], []
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            preds_logits = tta_model(images)              # TTA-averaged logits
            preds_prob = torch.sigmoid(preds_logits)

            if criterion is not None:
                try:
                    losses.append(float(criterion(preds_logits, masks).item()))
                except Exception:
                    pass

            for i in range(preds_prob.shape[0]):
                p = preds_prob[i, 0].cpu().numpy()
                gt = masks[i, 0].cpu().numpy()

                p_clean = post_process_mask(p, threshold=0.5)

                p_sum = float(p_clean.sum())
                gt_sum = float(gt.sum())
                if p_sum == 0 and gt_sum == 0:
                    dice_scores.append(1.0)
                    iou_scores.append(1.0)
                    continue

                inter = float((p_clean * gt).sum())
                dice = 2.0 * inter / (p_sum + gt_sum + 1e-6)
                iou = inter / (p_sum + gt_sum - inter + 1e-6)
                dice_scores.append(dice)
                iou_scores.append(iou)

    mean_loss = float(np.mean(losses)) if losses else float('nan')
    return mean_loss, float(np.mean(dice_scores)), float(np.mean(iou_scores))


print("EMA + TTA-aware validation utilities ready.")
print(f"  TTA transforms: {len(TTA_TRANSFORMS)} augmentations averaged per image")


## 6. Data Preparation

In [ ]:
# Split data into train/val/test at patient level (avoids patient leakage)
all_patients = sorted({r['patient_id'] for r in sample_records})

train_patients, temp_patients = train_test_split(
    all_patients, test_size=0.3, random_state=42
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.4, random_state=42
)

# Aggressive dataset subsampling targets
MAX_TRAIN_PATIENTS = 120
MAX_VAL_PATIENTS = 30
MAX_TEST_PATIENTS = 20

rng = np.random.default_rng(42)

def cap_patients(patient_list, max_count):
    patient_list = sorted(patient_list)
    if len(patient_list) <= max_count:
        return patient_list
    idx = rng.choice(len(patient_list), size=max_count, replace=False)
    return [patient_list[i] for i in sorted(idx)]

train_patients = cap_patients(train_patients, MAX_TRAIN_PATIENTS)
val_patients = cap_patients(val_patients, MAX_VAL_PATIENTS)
test_patients = cap_patients(test_patients, MAX_TEST_PATIENTS)

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

# Leakage sanity checks
print(f"Patient overlap train∩val: {len(train_set.intersection(val_set))}")
print(f"Patient overlap train∩test: {len(train_set.intersection(test_set))}")
print(f"Patient overlap val∩test: {len(val_set.intersection(test_set))}")

# ── Split records by patient ─────────────────────────────────────────────────
all_train_records = [r for r in sample_records if r['patient_id'] in train_set]
val_records = [r for r in sample_records if r['patient_id'] in val_set]
test_records = [r for r in sample_records if r['patient_id'] in test_set]

# ── KEY FIX: Subsample empty-mask slices in training ─────────────────────────
# Problem: ~70-80% of slices have no tumor at all. The model spends most of
# its gradient budget learning "output nothing" instead of tumor boundaries.
# Fix: keep ALL tumor slices + only ~20% of empty-mask slices.
EMPTY_KEEP_RATIO = 0.40  # raised from 0.20 — more 'negative examples' improves FP control

tumor_records = [r for r in all_train_records if r['has_tumor']]
empty_records = [r for r in all_train_records if not r['has_tumor']]

n_empty_keep = max(1, int(len(empty_records) * EMPTY_KEEP_RATIO))
empty_keep_idx = rng.choice(len(empty_records), size=n_empty_keep, replace=False)
empty_kept = [empty_records[i] for i in sorted(empty_keep_idx)]

train_records = tumor_records + empty_kept
rng.shuffle(train_records)

print(f"\nTraining slice filtering:")
print(f"  Tumor slices (all kept):   {len(tumor_records)}")
print(f"  Empty slices (kept {EMPTY_KEEP_RATIO:.0%}): {n_empty_keep} / {len(empty_records)}")
print(f"  Final training samples:    {len(train_records)}")
pct_tumor = 100 * len(tumor_records) / len(train_records)
print(f"  Tumor fraction in train:   {pct_tumor:.1f}%")

print(f"\nValidation samples: {len(val_records)} (all slices, no filtering)")
print(f"Test samples: {len(test_records)} (all slices, no filtering)")
print(f"Patients (train/val/test): {len(train_set)}/{len(val_set)}/{len(test_set)}")

# Create datasets
IMG_SIZE = 192
BATCH_SIZE = 12 if torch.cuda.is_available() else 4

# ── Strengthened augmentation (v3: heavier spatial + intensity diversity) ─────
# With very few patients, the model memorises patient-specific anatomy fast.
# Aggressive augmentation acts as implicit data multiplication.
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    # Geometric — heavier spatial warping
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=25, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.12, scale_limit=0.20, rotate_limit=20, p=0.6),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.25),

    # Intensity — wider range + extra channel-level perturbation
    A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.GaussNoise(var_limit=(0.001, 0.008), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),

    # Regularization: randomly drop rectangular regions (cutout)
    A.CoarseDropout(max_holes=8, max_height=24, max_width=24,
                    min_holes=2, min_height=8, min_width=8,
                    fill_value=0, p=0.4),

    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])

# Validation/Test: resize + normalize only
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])

print("\nBuilding 2D slice datasets (volumes are pre-loaded eagerly)...")
train_dataset = BrainTumorDataset(
    train_records, transform=train_transform, img_size=IMG_SIZE, patch_transform=None,
)
val_dataset = BrainTumorDataset(
    val_records, transform=val_transform, img_size=IMG_SIZE,
)
test_dataset = BrainTumorDataset(
    test_records, transform=val_transform, img_size=IMG_SIZE,
)

# DataLoader configuration
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: batch={train_loader.batch_size}, samples={len(train_dataset)}")
print(f"val_loader:   batch={val_loader.batch_size}, samples={len(val_dataset)}")
print(f"test_loader:  batch={test_loader.batch_size}, samples={len(test_dataset)}")

## 7. Model Initialization

In [ ]:
# Initialize compact nnU-Net 2D model (v3: halved channels, full-path dropout)
model = nnUNet2D(
    in_channels=4,
    num_classes=1,
    deep_supervision=True,
    drop_rate=0.3,
)

# Multi-GPU support
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(device)

# Loss function: deep supervision wrapper around DiceCE + Focal
base_loss = OptimalBrainTumorLoss()
ds_criterion = DeepSupervisionLoss(base_loss=base_loss, weights=(1.0, 0.5, 0.25))
# Add a mild false-positive penalty on empty-GT slices (full-res output only).
criterion = CriterionWithEmptyPenalty(ds_criterion, penalty_weight=0.1)

# AdamW: lower LR + stronger weight decay to combat overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

# Mixed precision scaler (CUDA only)
scaler = GradScaler(device='cuda', enabled=torch.cuda.is_available())
print(f"AMP enabled: {scaler.is_enabled()}")

# Canonical model paths
BEST_MODEL_PATH = 'best_nnunet2d.pth'        # holds EMA weights of the best validation Dice
CHECKPOINT_PATH = 'checkpoint_nnunet2d.pth'
FINAL_MODEL_PATH = 'nnunet2d_final.pth'

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Cosine annealing with warm restarts — smoother LR decay than ReduceLROnPlateau,
# prevents the model from settling into sharp (overfitting) minima.
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6,
)
print(f"Scheduler: CosineAnnealingWarmRestarts (T_0=20, T_mult=2, eta_min=1e-6)")

# ── EMA shadow weights ───────────────────────────────────────────────────────
# decay=0.999 means the EMA averages roughly the last ~1000 optimizer steps.
# For ~600 steps/epoch this gives a smooth window of ~1.5 epochs.
ema = ModelEMA(model, decay=0.999)
print(f"EMA initialized (decay={ema.decay})")


# ── Snapshot ensembling configuration ────────────────────────────────────────
# CosineAnnealingWarmRestarts: with T_0=20, T_mult=2 the cycle ends sit at
# epochs 19, 59, 139, … Each cycle ends at a *local minimum* of the loss
# landscape (LR=eta_min), giving a good ensemble member.
def _compute_snapshot_epochs(t_0: int, t_mult: int, num_epochs: int):
    epochs, cycle_end, cycle_len = [], t_0 - 1, t_0
    while cycle_end < num_epochs:
        epochs.append(cycle_end)
        cycle_len *= t_mult
        cycle_end += cycle_len
    return epochs

SNAPSHOT_EPOCHS = _compute_snapshot_epochs(t_0=20, t_mult=2, num_epochs=80)
SNAPSHOT_PATH_FMT = "snapshot_{idx}.pth"
print(f"Snapshot epochs (0-indexed): {SNAPSHOT_EPOCHS}")


## 8. Training Loop

In [ ]:
def _unwrap_logits(outputs):
    """Return full-resolution logits when deep supervision returns a tuple."""
    if isinstance(outputs, (tuple, list)):
        return outputs[0]
    return outputs


def train_epoch(model, dataloader, criterion, optimizer, device, scaler, epoch=0, ema=None):
    """Train for one epoch with optional AMP and optional EMA weight tracking."""
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    nan_count = 0
    amp_enabled = scaler.is_enabled()
    amp_device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

    for batch_idx, (images, masks) in enumerate(dataloader):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        with autocast(device_type=amp_device_type, enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, masks)

        # NaN guard: skip batch AND zero grads to prevent optimizer corruption
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        # Backward + optimizer step
        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # Update EMA after every successful optimizer step
        if ema is not None:
            ema.update(model)

        # Metrics from full-resolution logits
        logits = _unwrap_logits(outputs)
        dice = dice_coefficient(logits, masks)
        iou = iou_score(logits, masks)

        running_loss += loss.item()
        running_dice += dice
        running_iou += iou

    valid_batches = len(dataloader) - nan_count
    if valid_batches == 0:
        print(f"  WARNING: All {len(dataloader)} batches produced NaN!")
        return float('nan'), 0.0, 0.0

    if nan_count > 0:
        print(f"  NaN batches skipped: {nan_count}/{len(dataloader)}")

    epoch_loss = running_loss / valid_batches
    epoch_dice = running_dice / valid_batches
    epoch_iou = running_iou / valid_batches

    return epoch_loss, epoch_dice, epoch_iou


def validate_epoch(model, dataloader, criterion, device):
    """Plain validation pass (no TTA) — kept for backward compatibility / quick checks."""
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, masks)
            logits = _unwrap_logits(outputs)

            dice = dice_coefficient(logits, masks)
            iou = iou_score(logits, masks)

            running_loss += loss.item()
            running_dice += dice
            running_iou += iou

    epoch_loss = running_loss / len(dataloader)
    epoch_dice = running_dice / len(dataloader)
    epoch_iou = running_iou / len(dataloader)

    return epoch_loss, epoch_dice, epoch_iou


In [ ]:
# Training configuration
NUM_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 8        # measured in *validation* events
CHECKPOINT_INTERVAL = 5
VALIDATE_EVERY = 5                 # TTA validation is ~8x slower → validate less often

# Set to True when you want to force a fresh run
RESET_CHECKPOINT = True
if RESET_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f"Removed old checkpoint: {CHECKPOINT_PATH}")

# Training history
history = {
    'train_loss': [], 'train_dice': [], 'train_iou': [],
    'val_loss': [], 'val_dice': [], 'val_iou': []
}

best_val_dice = 0.0
early_stopping_counter = 0
start_epoch = 0

# Load checkpoint if exists
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'ema_state_dict' in checkpoint:
        ema.module.load_state_dict(checkpoint['ema_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_dice = checkpoint['best_val_dice']
    early_stopping_counter = checkpoint['early_stopping_counter']
    print(f"Resumed from epoch {start_epoch}")
    print(f"Best validation Dice (TTA+postproc) so far: {best_val_dice:.4f}\n")
else:
    print("No checkpoint found. Starting training from scratch...\n")

print("Starting training...")
print("Validation uses EMA weights + TTA + post-processing (matches deployment).\n")

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)

    # Train every epoch (EMA updated per optimizer step inside train_epoch)
    train_loss, train_dice, train_iou = train_epoch(
        model, train_loader, criterion, optimizer, device, scaler, epoch, ema=ema
    )

    # Cosine annealing steps every epoch (not tied to validation)
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]['lr']

    # Save train history every epoch
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['train_iou'].append(train_iou)

    # Validate every N epochs — TTA is expensive
    do_validate = (epoch % VALIDATE_EVERY == 0)
    if do_validate:
        # Validate the EMA shadow weights, not the live model
        val_loss, val_dice, val_iou = validate_with_tta_postproc(
            ema.module, val_loader, device, criterion=criterion,
        )

        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_iou'].append(val_iou)

        print(f"\nTrain Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f} | Train IoU: {train_iou:.4f}")
        print(f"Val (EMA+TTA+postproc) Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")
        print(f"LR: {current_lr:.2e}")

        # Save EMA weights as the best model — that's what we'll deploy
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(ema.module.state_dict(), BEST_MODEL_PATH)
            print(f"Best model saved at {BEST_MODEL_PATH} (EMA+TTA Dice: {best_val_dice:.4f})")
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1
    else:
        print(f"\nTrain Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f} | Train IoU: {train_iou:.4f}")
        print(f"LR: {current_lr:.2e} | Skipping validation this epoch (every {VALIDATE_EVERY} epochs)")

    # Save EMA snapshot at the end of each cosine cycle (free ensemble member)
    if epoch in SNAPSHOT_EPOCHS:
        snap_idx = SNAPSHOT_EPOCHS.index(epoch)
        snap_path = SNAPSHOT_PATH_FMT.format(idx=snap_idx)
        torch.save(ema.module.state_dict(), snap_path)
        print(f"📸 Snapshot {snap_idx} saved at {snap_path} (epoch {epoch+1}, end of cosine cycle)")

    # Save checkpoint periodically
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.module.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'best_val_dice': best_val_dice,
            'early_stopping_counter': early_stopping_counter,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"Checkpoint saved at epoch {epoch+1}")

    # Early stopping (evaluated only on validation epochs)
    if do_validate and early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        print("=" * 50)
        break

print(f"Training completed. Best Val Dice (EMA+TTA+postproc): {best_val_dice:.4f}")
print("=" * 50)


## 9. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Dice Score
axes[1].plot(history['train_dice'], label='Train Dice')
axes[1].plot(history['val_dice'], label='Val Dice')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Training and Validation Dice Score')
axes[1].legend()
axes[1].grid(True)

# IoU Score
axes[2].plot(history['train_iou'], label='Train IoU')
axes[2].plot(history['val_iou'], label='Val IoU')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('IoU Score')
axes[2].set_title('Training and Validation IoU Score')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Test Set Evaluation

In [ ]:
# Load best model
_target = model.module if isinstance(model, torch.nn.DataParallel) else model
_target.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

# Evaluate on test set
test_loss, test_dice, test_iou = validate_epoch(model, test_loader, criterion, device)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Model: {BEST_MODEL_PATH}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice Score: {test_dice:.4f}")
print(f"Test IoU Score: {test_iou:.4f}")
print("="*50)

## 10.1. Test-Time Augmentation (TTA) + Post-Processing

Apply TTA and morphological post-processing to improve predictions

In [ ]:
import ttach as tta
from scipy import ndimage

# Load best model
_target = model.module if isinstance(model, torch.nn.DataParallel) else model
_target.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

print("="*60)
print("SETTING UP TTA + POST-PROCESSING")
print("="*60)

# === TTA Configuration ===
tta_transforms = tta.Compose([
    tta.HorizontalFlip(),
    tta.VerticalFlip(),
    tta.Rotate90(angles=[0, 90, 180, 270]),
])

tta_model = tta.SegmentationTTAWrapper(
    model,
    tta_transforms,
    merge_mode='mean'
)

print("TTA model configured with transforms:")
print("  - Horizontal Flip")
print("  - Vertical Flip")
print("  - Rotation (0, 90, 180, 270 degrees)")
print(f"  - Total augmentations: {len(tta_transforms)}")
print("="*60 + "\n")

In [ ]:
# === POST-PROCESSING FUNCTION ===
def post_process(pred_prob, threshold=0.5):
    """
    Clean predictions using morphological operations
    
    Args:
        pred_prob: Probability map (H, W) with values [0, 1]
        threshold: Binary threshold (default: 0.5)
    
    Returns:
        Cleaned binary mask (H, W)
    """
    # Threshold to binary
    pred_binary = (pred_prob > threshold).astype(np.uint8)
    
    # Morphological opening (remove small noise)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    pred_binary = cv2.morphologyEx(pred_binary, cv2.MORPH_OPEN, kernel)
    
    # Morphological closing (fill small gaps)
    pred_binary = cv2.morphologyEx(pred_binary, cv2.MORPH_CLOSE, kernel)
    
    # Fill holes inside tumor regions
    pred_binary = ndimage.binary_fill_holes(pred_binary).astype(np.uint8)
    
    # Keep only largest connected component (main tumor)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_binary, connectivity=8)
    if num_labels > 1:
        # Get largest component (ignore background label 0)
        largest_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
        pred_binary = (labels == largest_label).astype(np.uint8)
    
    return pred_binary

print("✓ Post-processing function defined")
print("  - Morphological opening (noise removal)")
print("  - Morphological closing (gap filling)")
print("  - Hole filling")
print("  - Largest component extraction")

In [ ]:
# === EVALUATE WITH TTA + POST-PROCESSING ===
def evaluate_tta_postprocess(tta_model, dataloader, device):
    """
    CORRECTED TTA evaluation: Threshold AFTER averaging predictions
    
    Args:
        tta_model: TTA-wrapped model
        dataloader: Test/validation dataloader
        device: Device to run on
    
    Returns:
        Average Dice score and IoU score
    """
    dice_scores = []
    iou_scores = []
    
    print("\nEvaluating with TTA + Post-processing...")
    
    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Processing batches"):
            images = images.to(device)
            masks = masks.to(device)
            
            # TTA prediction: Multiple augmentations averaged FIRST
            preds_tta = tta_model(images)  # [B, 1, H, W] - averaged logits
            
            # Threshold AFTER TTA averaging (critical!)
            preds_prob = torch.sigmoid(preds_tta)
            
            # Apply post-processing to averaged predictions
            for i in range(preds_prob.shape[0]):
                pred_prob_2d = preds_prob[i, 0].cpu().numpy()  # (H, W)
                mask_2d = masks[i, 0].cpu().numpy()  # (H, W)
                
                # Post-process the averaged prediction
                pred_clean = post_process(pred_prob_2d, threshold=0.5)
                
                # Calculate metrics
                dice = dice_coefficient(
                    torch.from_numpy(pred_clean).unsqueeze(0).unsqueeze(0).float(),
                    torch.from_numpy(mask_2d).unsqueeze(0).unsqueeze(0).float()
                )
                
                iou = iou_score(
                    torch.from_numpy(pred_clean).unsqueeze(0).unsqueeze(0).float(),
                    torch.from_numpy(mask_2d).unsqueeze(0).unsqueeze(0).float()
                )
                
                dice_scores.append(dice)
                iou_scores.append(iou)
    
    return np.mean(dice_scores), np.mean(iou_scores)

print("✓ TTA + Post-processing evaluation function ready (CORRECTED)")
print("  - TTA averages predictions BEFORE thresholding")
print("  - Post-processing applied to averaged predictions")

## 10.2. Test Set Evaluation with TTA + Post-Processing

In [ ]:
# Evaluate on test set with TTA + Post-processing
print("\n" + "="*60)
print("TEST SET EVALUATION WITH TTA + POST-PROCESSING")
print("="*60)

tta_test_dice, tta_test_iou = evaluate_tta_postprocess(tta_model, test_loader, device)

print("\n" + "="*60)
print("COMPARISON: BASELINE vs TTA + POST-PROCESSING")
print("="*60)
print(f"\nBaseline Results:")
print(f"  Test Dice: {test_dice:.4f}")
print(f"  Test IoU:  {test_iou:.4f}")

print(f"\nTTA + Post-Processing Results:")
print(f"  Test Dice: {tta_test_dice:.4f} (Δ {tta_test_dice - test_dice:+.4f})")
print(f"  Test IoU:  {tta_test_iou:.4f} (Δ {tta_test_iou - test_iou:+.4f})")

if tta_test_dice > test_dice:
    improvement = ((tta_test_dice - test_dice) / test_dice) * 100
    print(f"\n✅ TTA + Post-processing improved Dice by {improvement:.2f}%")
else:
    print(f"\n⚠️  No improvement from TTA + Post-processing")

print("="*60)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  Snapshot Ensemble + TTA + Post-Processing  (best-quality inference)
#
#  For each saved snapshot (one per cosine restart):
#    1. Load EMA weights into the model
#    2. Run TTA-averaged inference over the entire test set
#    3. Store sigmoid probabilities
#  Then average the probability maps across snapshots and apply post-processing.
#  This is a "free ensemble" — uses checkpoints from a single training run.
# ──────────────────────────────────────────────────────────────────────────────
import os
import glob


def evaluate_snapshot_ensemble(model_template, snapshot_paths, dataloader, device,
                                threshold=0.5, use_tta=True):
    """Snapshot ensemble + (optional) TTA + post-processing."""
    base = model_template.module if isinstance(model_template, torch.nn.DataParallel) else model_template
    per_snapshot_probs, all_masks = [], []

    for snap_idx, snap_path in enumerate(snapshot_paths):
        print(f"\n  Loading snapshot {snap_idx+1}/{len(snapshot_paths)}: {snap_path}")
        base.load_state_dict(torch.load(snap_path, map_location=device))
        base.eval()

        if use_tta:
            wrapped = _DSStripWrapper(base).to(device)
            inf_model = tta_lib.SegmentationTTAWrapper(wrapped, TTA_TRANSFORMS, merge_mode='mean')
        else:
            inf_model = _DSStripWrapper(base).to(device)

        snap_probs = []
        with torch.no_grad():
            for batch_idx, (images, masks) in enumerate(tqdm(dataloader, desc=f"snap{snap_idx+1}")):
                images = images.to(device, non_blocking=True)
                logits = inf_model(images)
                probs = torch.sigmoid(logits).cpu()
                snap_probs.append(probs)
                if snap_idx == 0:
                    all_masks.append(masks)
        per_snapshot_probs.append(torch.cat(snap_probs, dim=0))

    masks_cat = torch.cat(all_masks, dim=0)
    avg_probs = torch.stack(per_snapshot_probs, dim=0).mean(dim=0)   # (N, 1, H, W)

    dice_scores, iou_scores = [], []
    for i in range(avg_probs.shape[0]):
        p = avg_probs[i, 0].numpy()
        gt = masks_cat[i, 0].numpy()
        p_clean = post_process_mask(p, threshold=threshold)

        p_sum = float(p_clean.sum())
        gt_sum = float(gt.sum())
        if p_sum == 0 and gt_sum == 0:
            dice_scores.append(1.0); iou_scores.append(1.0); continue
        inter = float((p_clean * gt).sum())
        dice = 2.0 * inter / (p_sum + gt_sum + 1e-6)
        iou = inter / (p_sum + gt_sum - inter + 1e-6)
        dice_scores.append(dice); iou_scores.append(iou)

    return float(np.mean(dice_scores)), float(np.mean(iou_scores))


# ── Locate snapshot files ─────────────────────────────────────────────────────
snapshot_paths = sorted(glob.glob("snapshot_*.pth"))
print("=" * 60)
print("SNAPSHOT ENSEMBLE EVALUATION")
print("=" * 60)
print(f"Found {len(snapshot_paths)} snapshot(s):")
for sp in snapshot_paths:
    print(f"  • {sp}")

if len(snapshot_paths) >= 1:
    # Use the unwrapped model as the template (state_dicts are unwrapped)
    _template = model.module if isinstance(model, torch.nn.DataParallel) else model

    ensemble_dice, ensemble_iou = evaluate_snapshot_ensemble(
        _template, snapshot_paths, test_loader, device, threshold=0.5, use_tta=True,
    )

    print("\n" + "=" * 60)
    print("COMPARISON: BASELINE  vs  TTA+POSTPROC  vs  SNAPSHOT ENSEMBLE")
    print("=" * 60)
    print(f"Baseline:                Dice = {test_dice:.4f} | IoU = {test_iou:.4f}")
    print(f"TTA + Post-processing:   Dice = {tta_test_dice:.4f} | IoU = {tta_test_iou:.4f}")
    print(f"Snapshot Ensemble + TTA: Dice = {ensemble_dice:.4f} | IoU = {ensemble_iou:.4f}")
    delta_vs_tta = ensemble_dice - tta_test_dice
    delta_vs_base = ensemble_dice - test_dice
    print(f"\nΔ vs TTA-only:   {delta_vs_tta:+.4f}")
    print(f"Δ vs baseline:   {delta_vs_base:+.4f}")
    print("=" * 60)

    # Restore the best single-checkpoint weights so downstream cells (Grad-CAM,
    # visualization) behave consistently.
    _template.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print("\nRestored best single-checkpoint weights for downstream cells.")
else:
    print("No snapshots found — re-train to populate snapshot_*.pth files,")
    print("or skip this cell.")


## 11. Prediction Visualization

In [ ]:
def analyze_split_distribution(loader, name):
    """Analyze tumor size distribution in a dataset split"""
    tumor_sizes = []
    
    for images, masks in loader:
        # masks shape: [B, 1, H, W]
        masks_np = masks.numpy()
        for mask in masks_np:
            tumor_pixels = np.sum(mask)
            tumor_sizes.append(tumor_pixels)
    
    # Categorize by size
    empty = sum(1 for x in tumor_sizes if x == 0)
    tiny = sum(1 for x in tumor_sizes if 0 < x < 100)
    small = sum(1 for x in tumor_sizes if 100 <= x < 500)
    medium = sum(1 for x in tumor_sizes if 500 <= x < 2000)
    large = sum(1 for x in tumor_sizes if x >= 2000)
    
    total = len(tumor_sizes)
    
    print(f"\n{name} Distribution:")
    print(f"  Empty (0):        {empty:4d} ({100*empty/total:5.1f}%)")
    print(f"  Tiny (<100):      {tiny:4d} ({100*tiny/total:5.1f}%)")
    print(f"  Small (100-500):  {small:4d} ({100*small/total:5.1f}%)")
    print(f"  Medium (500-2k):  {medium:4d} ({100*medium/total:5.1f}%)")
    print(f"  Large (>2k):      {large:4d} ({100*large/total:5.1f}%)")
    print(f"  Total:            {total:4d}")

print("="*60)
print("DATASET SPLIT ANALYSIS")
print("="*60)
analyze_split_distribution(val_loader, "Validation")
analyze_split_distribution(test_loader, "Test")
print("="*60)

In [ ]:
def visualize_predictions(model, dataloader, device, num_samples=8):
    """Visualize model predictions"""
    model.eval()
    
    images_list = []
    masks_list = []
    preds_list = []
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs) > 0.5
            
            images_list.append(images.cpu())
            masks_list.append(masks.cpu())
            preds_list.append(preds.cpu())
            
            if len(images_list) * images.size(0) >= num_samples:
                break
    
    images = torch.cat(images_list, dim=0)[:num_samples]
    masks = torch.cat(masks_list, dim=0)[:num_samples]
    preds = torch.cat(preds_list, dim=0)[:num_samples]
    
    # Plot
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    
    for i in range(num_samples):
        # Original image (FLAIR channel)
        img = images[i, 1].numpy()  # FLAIR is channel 1
        axes[i, 0].imshow(img, cmap='gray')
        axes[i, 0].set_title('Input (FLAIR)')
        axes[i, 0].axis('off')
        
        # Ground truth
        mask = masks[i, 0].numpy()
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        pred = preds[i, 0].numpy()
        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        overlay = np.stack([img, img, img], axis=-1)
        overlay[pred > 0] = [1, 0, 0]  # Red for prediction
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize predictions on test set
visualize_predictions(model, test_loader, device, num_samples=8)

## 13. Grad-CAM Explainability

Two complementary views of what the model has learned:

| Panel | What it shows |
|---|---|
| **Focus map** | A single heat-map overlay on the FLAIR image showing *where in the image* the model pays attention when making its segmentation decision. High-intensity regions (red) drove the model's output the most. |
| **Layer activation maps** | The same Grad-CAM computed at **four encoder depths** (shallow → bottleneck) and one **decoder** stage, revealing how feature abstraction progresses through the network. Early layers respond to low-level edges; deep layers respond to higher-level tumour context. |

### Implementation notes
* **No external library required** — hook-based Grad-CAM implemented from scratch.  
* **Scalar for backprop**: `sum(sigmoid(logits) × predicted_mask)` — this focuses the gradient signal on the predicted tumour region, producing sharper and more informative maps.
* **Target layers** are automatically discovered by scanning `model.encoder.named_modules()` for the last `Conv2d` in each encoder quarter, plus the final decoder block.


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# ─────────────────────────────────────────────────────────────────────────────
#  Grad-CAM engine (hook-based, no external library)
# ─────────────────────────────────────────────────────────────────────────────

class SegGradCAM:
    """Gradient-weighted Class Activation Mapping for segmentation models.

    Registers one forward hook and one backward hook on ``target_layer``.
    Call ``__call__`` to compute the CAM for a single image.

    Parameters
    ----------
    model        : nn.Module  — the segmentation model (eval mode recommended)
    target_layer : nn.Module  — the layer to probe (must produce a 4-D feature map)
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self._acts: torch.Tensor | None = None
        self._grads: torch.Tensor | None = None

        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, _m, _inp, out):
        self._acts = out.detach()

    def _save_grad(self, _m, _inp, grad_out):
        self._grads = grad_out[0].detach()

    def remove(self):
        """Remove hooks — call when done to free memory."""
        self._fwd.remove()
        self._bwd.remove()

    @torch.enable_grad()
    def __call__(
        self,
        image: torch.Tensor,             # (1, C, H, W)  on device
        roi_mask: torch.Tensor | None = None,  # (1, 1, H, W) optional focus region
    ) -> np.ndarray:
        """Return a [0,1]-normalised Grad-CAM heat-map (H×W numpy array)."""
        self.model.eval()
        self._acts = self._grads = None

        # ── Forward pass ─────────────────────────────────────────────────
        out = self.model(image)                        # (1, 1, H, W) logits
        prob = torch.sigmoid(out)

        # ── Scalar objective ─────────────────────────────────────────────
        # Back-prop signal: sum of predicted probability over the tumour
        # region predicted by the model (self-supervised → no GT needed).
        pred_mask = (prob > 0.5).float()
        if roi_mask is not None:
            roi = roi_mask.to(image.device)
            score = (prob * roi).sum()
        elif pred_mask.sum() > 0:
            score = (prob * pred_mask).sum()
        else:
            # No prediction — fall back to summing the whole map
            score = prob.sum()

        # ── Backward pass ────────────────────────────────────────────────
        self.model.zero_grad()
        score.backward()

        if self._grads is None or self._acts is None:
            return np.zeros(image.shape[-2:])

        # ── Grad-CAM formula ─────────────────────────────────────────────
        # α_k = (1/Z) Σ_{i,j} ∂score/∂A^k_{ij}   (global average pool)
        weights = self._grads.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self._acts).sum(dim=1)               # (1, H', W')
        cam = F.relu(cam).squeeze(0).cpu().numpy()            # (H', W')

        # ── Resize to input resolution ────────────────────────────────────
        H, W = image.shape[2], image.shape[3]
        if cam.shape == ():            # scalar edge case
            cam = np.full((H, W), float(cam))
        elif cam.ndim == 0:
            cam = np.full((H, W), float(cam))
        else:
            cam = cv2.resize(cam, (W, H), interpolation=cv2.INTER_LINEAR)

        # ── Normalise ─────────────────────────────────────────────────────
        cmin, cmax = cam.min(), cam.max()
        if cmax > cmin:
            cam = (cam - cmin) / (cmax - cmin)

        return cam


# ─────────────────────────────────────────────────────────────────────────────
#  Layer discovery — finds representative Conv2d layers at n encoder depths
# ─────────────────────────────────────────────────────────────────────────────

def discover_encoder_layers(model: nn.Module, n_depths: int = 4):
    """Return [(label, Conv2d), …] at ``n_depths`` evenly-spaced encoder depths.

    Works with any SMP encoder by scanning ``model.encoder.named_modules()``.
    Targets the last Conv2d in each n-th segment of the full module list,
    which reliably captures progressively deeper features without relying
    on hard-coded layer names.
    """
    convs = [
        (name, mod)
        for name, mod in model.encoder.named_modules()
        if isinstance(mod, nn.Conv2d)
    ]
    if not convs:
        raise RuntimeError("No Conv2d found in model.encoder — check SMP version.")

    n = len(convs)
    result = []
    for i in range(n_depths):
        idx  = min(int((i + 1) * n / n_depths) - 1, n - 1)
        name, mod = convs[idx]
        # Human-readable depth label
        parts = name.split(".")
        short = ".".join(parts[:min(4, len(parts))])
        label = f"Encoder depth {i + 1}  [{short}]"
        result.append((label, mod))
    return result


def get_decoder_layer(model: nn.Module):
    """Return (label, Conv2d) from the last/shallowest decoder block.

    SMP UNet++ stores decoder blocks in ``model.decoder.blocks``.
    The *last* block is closest to the full-resolution output and encodes
    the most semantically rich, spatially precise features.
    """
    for name, mod in reversed(list(model.decoder.named_modules())):
        if isinstance(mod, nn.Conv2d):
            parts = name.split(".")
            short = ".".join(parts[:min(4, len(parts))])
            return (f"Decoder shallowest  [{short}]", mod)
    raise RuntimeError("No Conv2d found in model.decoder")


# ─────────────────────────────────────────────────────────────────────────────
#  Visualisation helpers
# ─────────────────────────────────────────────────────────────────────────────

CMAP_HEAT = cm.get_cmap("jet")     # continuous heat-map
ALPHA_BLEND = 0.55                  # heatmap-over-image opacity


def _flair(image_t: torch.Tensor) -> np.ndarray:
    """Extract FLAIR (channel 1) as a 2-D [0,1] numpy array."""
    img = image_t[0, 1].cpu().numpy()
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)


def _heatmap_overlay(flair_np: np.ndarray, cam: np.ndarray) -> np.ndarray:
    """Blend FLAIR greyscale image with a Jet colour-mapped CAM."""
    base   = np.stack([flair_np] * 3, axis=-1)               # (H,W,3)
    heat   = CMAP_HEAT(cam)[..., :3]                          # (H,W,3) RGBA→RGB
    blended = (1 - ALPHA_BLEND) * base + ALPHA_BLEND * heat
    return np.clip(blended, 0, 1)


def _add_colorbar(ax, vmin=0, vmax=1, label="GradCAM intensity"):
    """Append a colourbar to a matplotlib Axes."""
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sm   = cm.ScalarMappable(cmap=CMAP_HEAT, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label=label)


# ─────────────────────────────────────────────────────────────────────────────
#  Panel 1 — Focus map  (single target layer, N test samples)
# ─────────────────────────────────────────────────────────────────────────────

def visualize_gradcam_focus(
    model,
    dataloader,
    device,
    num_samples: int = 6,
    prefer_tumour: bool = True,
):
    """5-column panel for each sample:
        FLAIR | GT mask | Prediction | Raw CAM heat-map | CAM overlay on FLAIR

    Target layer: last decoder Conv2d (highest spatial resolution, most
    semantically meaningful for tumour localisation).
    """
    # ── Discover target layer ─────────────────────────────────────────────
    target_label, target_layer = get_decoder_layer(model)
    gcam = SegGradCAM(model, target_layer)
    print(f"[Focus] Target layer: {target_label}")

    # ── Collect samples ───────────────────────────────────────────────────
    samples = []
    for imgs, masks in dataloader:
        for i in range(imgs.size(0)):
            if prefer_tumour and masks[i].sum() < 10:
                continue          # skip near-empty slices
            samples.append((imgs[i:i+1], masks[i:i+1]))
            if len(samples) >= num_samples:
                break
        if len(samples) >= num_samples:
            break

    if not samples:         # fall back if no tumour samples found
        for imgs, masks in dataloader:
            for i in range(imgs.size(0)):
                samples.append((imgs[i:i+1], masks[i:i+1]))
                if len(samples) >= num_samples:
                    break
            if len(samples) >= num_samples:
                break

    # ── Plot ─────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        len(samples), 5,
        figsize=(22, 4.4 * len(samples)),
        gridspec_kw={"wspace": 0.05, "hspace": 0.35},
    )
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["FLAIR input", "Ground truth", "Prediction", "Grad-CAM heat-map", "CAM overlay"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=12, fontweight="bold", pad=8)

    for row, (img_t, mask_t) in enumerate(samples):
        img_dev = img_t.to(device)

        # Grad-CAM
        cam = gcam(img_dev)

        # Predictions
        with torch.no_grad():
            pred = (torch.sigmoid(model(img_dev)) > 0.5).float().cpu()

        flair = _flair(img_t)
        gt    = mask_t[0, 0].cpu().numpy()
        pr    = pred[0, 0].numpy()
        ovl   = _heatmap_overlay(flair, cam)

        axes[row, 0].imshow(flair, cmap="gray")
        axes[row, 1].imshow(gt,    cmap="gray")
        axes[row, 2].imshow(pr,    cmap="gray")
        axes[row, 3].imshow(cam,   cmap=CMAP_HEAT, vmin=0, vmax=1)
        axes[row, 4].imshow(ovl)

        # Dice for this sample
        intersection = (pr * gt).sum()
        dice_s = 2 * intersection / (pr.sum() + gt.sum() + 1e-8)
        axes[row, 4].set_xlabel(f"Dice = {dice_s:.3f}", fontsize=9)

        for ax in axes[row]:
            ax.axis("off")

    _add_colorbar(axes[-1, 3], label="Grad-CAM intensity")

    fig.suptitle(
        f"Grad-CAM Focus Maps  —  target: {target_label}",
        fontsize=14, fontweight="bold", y=1.01
    )
    plt.savefig("gradcam_focus.png", dpi=200, bbox_inches="tight")
    plt.show()
    gcam.remove()
    print("Saved: gradcam_focus.png")


# ─────────────────────────────────────────────────────────────────────────────
#  Panel 2 — Layer-by-layer activation maps across encoder depths
# ─────────────────────────────────────────────────────────────────────────────

def visualize_layer_gradcam(
    model,
    dataloader,
    device,
    n_encoder_depths: int = 4,
    num_samples: int = 4,
    prefer_tumour: bool = True,
):
    """Grid: rows = encoder depths + 1 decoder;  columns = test samples.

    Each cell shows the Grad-CAM heat-map computed with that specific layer
    as the target.  This reveals how feature abstraction progresses:
      * Depth 1  — low-level edges and intensity gradients
      * Depth 2  — tissue boundaries
      * Depth 3  — tumour shape context
      * Depth 4  — high-level semantic context (bottleneck)
      * Decoder  — final spatially-precise activation
    """
    # ── Discover target layers ────────────────────────────────────────────
    enc_layers = discover_encoder_layers(model, n_depths=n_encoder_depths)
    dec_label, dec_layer = get_decoder_layer(model)
    all_layers = enc_layers + [(dec_label, dec_layer)]
    n_layers   = len(all_layers)

    print("Target layers for layer-by-layer analysis:")
    for lbl, _ in all_layers:
        print(f"  • {lbl}")

    # ── Collect samples ───────────────────────────────────────────────────
    samples = []
    for imgs, masks in dataloader:
        for i in range(imgs.size(0)):
            if prefer_tumour and masks[i].sum() < 10:
                continue
            samples.append((imgs[i:i+1], masks[i:i+1]))
            if len(samples) >= num_samples:
                break
        if len(samples) >= num_samples:
            break

    if not samples:
        for imgs, masks in dataloader:
            for i in range(imgs.size(0)):
                samples.append((imgs[i:i+1], masks[i:i+1]))
                if len(samples) >= num_samples:
                    break
            if len(samples) >= num_samples:
                break

    # ── Plot ─────────────────────────────────────────────────────────────
    # rows = layers,  cols = samples
    FIG_W = max(5 * len(samples), 14)
    fig, axes = plt.subplots(
        n_layers, len(samples),
        figsize=(FIG_W, 3.8 * n_layers),
        gridspec_kw={"wspace": 0.05, "hspace": 0.45},
    )
    if n_layers == 1:
        axes = axes[np.newaxis, :]
    if len(samples) == 1:
        axes = axes[:, np.newaxis]

    for col, (img_t, _) in enumerate(samples):
        axes[0, col].set_title(f"Sample {col + 1}", fontsize=11, fontweight="bold", pad=6)

    for row, (lbl, layer) in enumerate(all_layers):
        gcam = SegGradCAM(model, layer)
        axes[row, 0].set_ylabel(
            lbl.split("[")[0].strip(),   # strip path for brevity
            fontsize=9, rotation=30, labelpad=70, va="center"
        )

        for col, (img_t, _) in enumerate(samples):
            img_dev = img_t.to(device)
            cam = gcam(img_dev)
            flair = _flair(img_t)
            ovl   = _heatmap_overlay(flair, cam)

            axes[row, col].imshow(ovl)
            axes[row, col].axis("off")

        gcam.remove()   # clean up hook before moving to next layer

    # Common colourbar
    _add_colorbar(
        axes[-1, -1],
        label="Grad-CAM  (0=inactive → 1=max activation)"
    )

    fig.suptitle(
        "Grad-CAM Layer-by-Layer Activation Maps\n"
        "(rows = encoder/decoder depth, columns = test samples)",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.savefig("gradcam_layers.png", dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved: gradcam_layers.png")


print("Grad-CAM engine loaded.")
print("  SegGradCAM         — hook-based Grad-CAM class")
print("  discover_encoder_layers() — auto-discovers encoder depth layers")
print("  visualize_gradcam_focus()   — focus heat-map panel")
print("  visualize_layer_gradcam()   — layer-by-layer activation panel")


In [ ]:
# Grad-CAM compatibility overrides for custom nnU-Net 2D

def _unwrap_cam_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def _last_conv2d(module: nn.Module) -> nn.Module:
    for m in reversed(list(module.modules())):
        if isinstance(m, nn.Conv2d):
            return m
    raise RuntimeError("No Conv2d found in module")


def discover_encoder_layers(model: nn.Module, n_depths: int = 4):
    """Return representative encoder layers for Grad-CAM.

    Supports both:
    - SMP-style models exposing model.encoder
    - custom nnUNet2D exposing enc1..enc4 + bottleneck
    """
    m = _unwrap_cam_model(model)

    if hasattr(m, 'encoder'):
        convs = [
            (name, mod)
            for name, mod in m.encoder.named_modules()
            if isinstance(mod, nn.Conv2d)
        ]
        if not convs:
            raise RuntimeError("No Conv2d found in model.encoder")

        n = len(convs)
        result = []
        for i in range(n_depths):
            idx = min(int((i + 1) * n / n_depths) - 1, n - 1)
            name, mod = convs[idx]
            result.append((f"Encoder depth {i + 1} [{name}]", mod))
        return result

    custom_blocks = [
        ('enc1', getattr(m, 'enc1', None)),
        ('enc2', getattr(m, 'enc2', None)),
        ('enc3', getattr(m, 'enc3', None)),
        ('enc4', getattr(m, 'enc4', None)),
        ('bottleneck', getattr(m, 'bottleneck', None)),
    ]
    custom_blocks = [(name, block) for name, block in custom_blocks if block is not None]
    if not custom_blocks:
        raise RuntimeError("Could not discover encoder layers for this model")

    picks = np.linspace(0, len(custom_blocks) - 1, num=n_depths, dtype=int)
    layers = []
    for i, idx in enumerate(picks):
        name, block = custom_blocks[int(idx)]
        layers.append((f"Encoder depth {i + 1} [{name}]", _last_conv2d(block)))
    return layers


def get_decoder_layer(model: nn.Module):
    """Return a decoder Conv2d layer for Grad-CAM overlays."""
    m = _unwrap_cam_model(model)

    if hasattr(m, 'decoder'):
        for name, mod in reversed(list(m.decoder.named_modules())):
            if isinstance(mod, nn.Conv2d):
                return (f"Decoder shallowest [{name}]", mod)

    if hasattr(m, 'dec1'):
        return ("Decoder shallowest [dec1]", _last_conv2d(m.dec1))

    raise RuntimeError("No suitable decoder Conv2d found for Grad-CAM")


print("Applied Grad-CAM compatibility overrides for nnU-Net 2D.")

In [ ]:
# -----------------------------------------------------------------------------
# Run Grad-CAM explainability
# -----------------------------------------------------------------------------

# Ensure the best model weights are loaded
_target = model.module if isinstance(model, torch.nn.DataParallel) else model
_target.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print("Best model weights loaded.\n")

# Panel 1 — Focus maps
print("=" * 60)
print("PANEL 1 — Grad-CAM Focus Maps")
print("  Target: last decoder Conv2d (highest-resolution, most semantic)")
print("=" * 60)

visualize_gradcam_focus(
    model=model,
    dataloader=test_loader,
    device=device,
    num_samples=6,
    prefer_tumour=True,
)

print()

# Panel 2 — Layer-by-layer activation maps
print("=" * 60)
print("PANEL 2 — Layer-by-Layer Grad-CAM Activation Maps")
print("  Rows = encoder depths (1=shallow -> 4=bottleneck) + decoder")
print("  Columns = test samples")
print("=" * 60)

visualize_layer_gradcam(
    model=model,
    dataloader=test_loader,
    device=device,
    n_encoder_depths=4,
    num_samples=4,
    prefer_tumour=True,
)

## 14. Save Final Model


In [ ]:
# Save final model with metadata
torch.save({
    'epoch': len(history['train_loss']),
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_dice': best_val_dice,
    'test_dice': test_dice,
    'test_iou': test_iou,
    'history': history,
}, FINAL_MODEL_PATH)

print(f"Final model saved successfully: {FINAL_MODEL_PATH}")

## 15. Model Summary


In [ ]:
print("\n" + "=" * 60)
print("nnU-Net 2D MODEL SUMMARY")
print("=" * 60)
print("Architecture: lightweight nnU-Net 2D with deep supervision")
print(f"Input Size: {IMG_SIZE}x{IMG_SIZE}x4 (4 MRI modalities)")
print("Output: Binary segmentation mask")
print(f"Total Parameters: {total_params:,}")
print(f"\nTraining Data: {len(train_records)} slices ({len(train_set)} patients)")
print(f"Validation Data: {len(val_records)} slices ({len(val_set)} patients)")
print(f"Test Data: {len(test_records)} slices ({len(test_set)} patients)")
print(f"\nBest Validation Dice: {best_val_dice:.4f}")
print(f"Test Dice Score: {test_dice:.4f}")
print(f"Test IoU Score: {test_iou:.4f}")
print(f"Best model path: {BEST_MODEL_PATH}")
print(f"Final model path: {FINAL_MODEL_PATH}")
print("=" * 60)